In [19]:
import json
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

print(f"numpy:       {np.__version__}")
print(f"pandas:      {pd.__version__}")
print(f"torch:       {torch.__version__}")

numpy:       2.2.6
pandas:      2.3.3
torch:       2.10.0+cu128


# 1. Check the annotation file to ensure nothing is strange

In [9]:
# load the annotation file
fn_annotation = '/storage/ice1/2/7/wzhu97/2026_freeman_bird_behavior/data/wz_data/H23/Annotations/instances_merged.json'
# fn_annotation = '/storage/ice1/2/7/wzhu97/2026_freeman_bird_behavior/data/wz_data/H03/Annotations/instances_merged.json'
print('#', '#'*50)
print(f"# ################## Checking {fn_annotation.split('/')[-3]} ##################")
print('#', '#'*50)

with open(fn_annotation, 'r') as f:
    coco = json.load(f)

# ########## 1. Check top level keys [PASSED] ##########
print('\n# ########## 1. Check top level keys ##########')
# a valid COCO file should have:
# dict_keys(['info', 'licenses', 'images', 'annotations', 'categories'])
print(coco.keys())

# ########## 2. Check images [PASSED] ##########
# - File height and width should be consistent across all images.
# - File names should match frame information
print('\n# ########## 2. Check images ##########')
print(f"# - Total images: {len(coco['images'])}")
print("# - Example of sample image entries:")
print(coco['images'][0])
print(coco['images'][-1])

# ########## 3. Check annotations [PASSED] ##########
print('\n# ########## 3. Check annotations ##########')
print(f"# - Total annotations: {len(coco['annotations'])}")
print("# - Sample annotation entries:")
print(coco['annotations'][0])
print(coco['annotations'][-1])


# ########## 4. Check categories [PASSED] ##########
print('\n# ########## 4. Check categories ##########')
print(coco['categories'])


# ########## 5. Check for common issues [PASSED] ##########
print('\n# ########## 5. Check for common issues ##########')
import numpy as np

# get all image IDs
image_ids = set([img['id'] for img in coco['images']])
anno_image_ids = set([ann['image_id'] for ann in coco['annotations']])

# check 1 - images with no annotations
unannotated = image_ids - anno_image_ids
print(f"# - N images with no annotations: {len(unannotated)}")

# check 2 - annotations pointing to non-existent images
orphan_annos = anno_image_ids - image_ids
print(f"# - N annotations with missing image_id: {len(orphan_annos)}")

# check 3 - invalid bounding boxes
issues = []
for ann in coco['annotations']:
    x, y, w, h = ann['bbox']
    if w <= 0 or h <= 0:
        issues.append(f"# - Zero/negative bbox size: annotation id {ann['id']}")
    if x < 0 or y < 0:
        issues.append(f"# - Negative bbox origin: annotation id {ann['id']}")

print(f"# - Bbox issues found: {len(issues)}")
for issue in issues[:10]:       # show first 10
    print(issue)

# check 4 - bounding boxes exceeding image dimensions
for ann in coco['annotations']:
    x, y, w, h = ann['bbox']
    # find corresponding image
    img = next(img for img in coco['images'] if img['id'] == ann['image_id'])
    if x + w > img['width'] or y + h > img['height']:
        print(f"# - Bbox exceeds image bounds: annotation id {ann['id']}")



# ########## 6. Qf ##########
print('\n# ########## 6. Quick summary stats ##########')
# annotations per image distribution
from collections import Counter
anno_counts = Counter([ann['image_id'] for ann in coco['annotations']])

print(f"# - Min annotations per image: {min(anno_counts.values())}")
print(f"# - Max annotations per image: {max(anno_counts.values())}")
print(f"# - Avg annotations per image: {np.mean(list(anno_counts.values())):.2f}")
print(f"# - Images with 0 annotations: {len(image_ids - set(anno_counts.keys()))}")
print(f"# - Images with >1 annotations: {len(np.array(list(anno_counts.keys()))[np.array(list(anno_counts.values()))>1])}")


# Check which images have >1 annotations (detected two objects?)


# ##################################################
# ################## Checking H23 ##################
# ##################################################

# ########## 1. Check top level keys ##########
dict_keys(['info', 'licenses', 'images', 'annotations', 'categories'])

# ########## 2. Check images ##########
# - Total images: 27860
# - Example of sample image entries:
{'id': 1, 'width': 1280, 'height': 720, 'file_name': 'IMG_0070_frame_000000.png', 'license': 0, 'flickr_url': '', 'coco_url': '', 'date_captured': 0}
{'id': 27860, 'width': 1280, 'height': 720, 'file_name': 'IMG_0069_frame_000302.png', 'license': 0, 'flickr_url': '', 'coco_url': '', 'date_captured': 0}

# ########## 3. Check annotations ##########
# - Total annotations: 11349
# - Sample annotation entries:
{'id': 1, 'image_id': 1, 'category_id': 1, 'segmentation': [], 'area': 10292.10250000001, 'bbox': [1177.5, 551.77, 101.45, 101.45], 'iscrowd': 0, 'attributes': {'occluded': False, 'rotation': 0.0, 'track_id': 

In [57]:
# Check which videos in H23 have positive detection (annotaiton)
# This will be used later to create a subset data for test runs

# load the annotation file
fn_annotation = '/storage/ice1/2/7/wzhu97/2026_freeman_bird_behavior/data/wz_data/H23/Annotations/instances_merged.json'

# load the annotation file
with open(fn_annotation, 'r') as f:
    coco = json.load(f)

anno_image_ids = set([ann['image_id'] for ann in coco['annotations']])
imgs_with_annotation = [img for img in coco['images'] if img['id'] in anno_image_ids]
# len(imgs_with_annotation)
# imgs_with_annotation

# Get video names
videos_with_annotation = set([img['file_name'].split('_frame_')[0] for img in imgs_with_annotation])
print(f'# N videos with annotations (positive detections): {len(videos_with_annotation)}')
# Majority of videos have annotations (85 out of 92)

# N videos with annotations (positive detections): 85


In [35]:
lst_image_ids = [15583, 15584, 15585, 15586, 15587, 15588, 19090, 19091, 19092,
                 19093, 19094, 19095, 19096, 19097, 19098, 19099, 19100, 19101,
                 19102, 19103, 21200, 21201, 21202, 21203, 21204, 21205, 21206,
                 21207, 21208, 21209, 21210, 21211]

[ann for ann in coco['annotations'] if ann['image_id'] in lst_image_ids]


[ann for ann in coco['annotations'] if ann['image_id'] in [15583]]



[{'id': 5325,
  'image_id': 15583,
  'category_id': 1,
  'segmentation': [],
  'area': 21369.1986,
  'bbox': [860.24, 244.49, 145.26, 147.11],
  'iscrowd': 0,
  'attributes': {'occluded': False,
   'rotation': 0.0,
   'track_id': 0,
   'keyframe': True}},
 {'id': 5326,
  'image_id': 15583,
  'category_id': 1,
  'segmentation': [],
  'area': 9429.512399999996,
  'bbox': [1190.38, 169.65, 89.32, 105.57],
  'iscrowd': 0,
  'attributes': {'occluded': False,
   'rotation': 0.0,
   'track_id': 1,
   'keyframe': True}}]

In [37]:
[img for img in coco['images'] if img['id']==15583]

[{'id': 15583,
  'width': 1280,
  'height': 720,
  'file_name': 'IMG_0193_frame_000140.png',
  'license': 0,
  'flickr_url': '',
  'coco_url': '',
  'date_captured': 0}]

# 2.  Create a subset list to use for test runs
* Already have lsit of frame files, just need to subset them
* Use H23 data since it contains balanced number of positive detection and no detections (ie no annotations)

In [55]:
sample_list = '/home/hice1/wzhu97/scratch/2026_freeman_bird_behavior/data/wz_data/H23/H23_frame_list.txt'
df = pd.read_csv(sample_list, header=None).rename(columns={0:'Path'})
df['Video'] = df['Path'].apply(lambda x: x.split('/')[-1].split('_frame_')[0])

# Select by frames
output_fn = '/home/hice1/wzhu97/scratch/2026_freeman_bird_behavior/data/wz_data/H23_frame_list.subset_0.01.csv'
df.sample(frac=0.01, random_state=42).to_csv(output_fn, index=False)

output_fn = '/home/hice1/wzhu97/scratch/2026_freeman_bird_behavior/data/wz_data/H23_frame_list.subset_0.1.csv'
df.sample(frac=0.1, random_state=42).to_csv(output_fn, index=False)

In [66]:
# Select by videos (do not put frams from the same video into train and test)
sample_list = '/home/hice1/wzhu97/scratch/2026_freeman_bird_behavior/data/wz_data/H23/H23_frame_list.txt'
df = pd.read_csv(sample_list, header=None).rename(columns={0:'Path'})
df['Video'] = df['Path'].apply(lambda x: x.split('/')[-1].split('_frame_')[0])
videos_no_annotation = set([v for v in df['Video'] if v not in videos_with_annotation])

# Select 5 videos, make sure to include one without annotation (no detection)
n_videos, n_positive_videos = 5, 4
rng = np.random.default_rng(seed=42)
videos_to_save = np.concatenate([rng.choice(list(videos_with_annotation), n_positive_videos), rng.choice(list(videos_no_annotation), n_videos-n_positive_videos)]) 
output_fn = f'/home/hice1/wzhu97/scratch/2026_freeman_bird_behavior/data/wz_data/H23_frame_list.subset_{n_videos}_videos.csv'
df[df['Video'].isin(videos_to_save)].to_csv(output_fn, index=False, header=False)

# Select 10 videos
n_videos, n_positive_videos = 10, 8
rng = np.random.default_rng(seed=42)

videos_to_save = np.concatenate([rng.choice(list(videos_with_annotation), n_positive_videos), rng.choice(list(videos_no_annotation), n_videos-n_positive_videos)]) 
output_fn = f'/home/hice1/wzhu97/scratch/2026_freeman_bird_behavior/data/wz_data/H23_frame_list.subset_{n_videos}_videos.csv'
df[df['Video'].isin(videos_to_save)].to_csv(output_fn, index=False, header=False)